# Перевод текста в числа (единый формат)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import json
import os
import shutil
import re
from datetime import datetime

class MedicalFeatureExtractor:
    """Извлечение признаков из медицинских JSON файлов"""
    
    def __init__(self):
        self.symptom_patterns = self._load_symptom_patterns()
        self.medical_terms = self._load_medical_terms()
        self.scaler = StandardScaler()
        
    def _load_symptom_patterns(self):
        """Паттерны для поиска симптомов"""
        return {
            'кашель': {
                'keywords': ['кашель', 'кашля', 'кашле', 'откашливан'],
                'intensity': {
                    'легкий': ['покашливан', 'легкий кашель', 'незначительный кашель'],
                    'средний': ['кашель', 'малопродуктивный', 'постоянный кашель'],
                    'сильный': ['сильный кашель', 'надсадный', 'мучительный', 'приступ кашля']
                }
            },
            'температура': {
                'keywords': ['температур', 'лихорадк', 'жар', 'озноб', 'гипертерми', 'субфебрильн', 'фебрильн'],
                'intensity': {
                    'легкая': ['субфебрильн', '37.', '37,'],
                    'средняя': ['температур', '38.', '38,', 'фебрильн'],
                    'высокая': ['высокая температур', '39.', '39,', '40.', '40,', 'гипертерми']
                }
            },
            'одышка': {
                'keywords': ['одышк', 'затруднен', 'дыхан', 'нехватк', 'воздух', 'удушь', 'диспноэ'],
                'intensity': {
                    'легкая': ['одышка при нагрузк', 'незначительная одышка'],
                    'средняя': ['одышк', 'затруднен', 'дыхан'],
                    'тяжелая': ['одышка в покое', 'сильная одышка', 'удушь']
                }
            },
            'слабость': {
                'keywords': ['слабост', 'усталост', 'недомогание', 'разбитост', 'астени', 'утомляемост'],
                'intensity': {
                    'легкая': ['слабост', 'недомогание'],
                    'средняя': ['выраженная слабост', 'сильная слабост'],
                    'тяжелая': ['резкая слабост', 'обездвиженност']
                }
            },
            'головная_боль': {
                'keywords': ['головная боль', 'головные боли', 'головной боль', 'цефалги', 'мигрен', 'болит голова'],
                'intensity': {
                    'легкая': ['головная боль', 'незначительная головная боль'],
                    'средняя': ['сильная головная боль', 'выраженная головная боль'],
                    'тяжелая': ['нестерпимая головная боль', 'мучительная головная боль']
                }
            },
            'рвота': {
                'keywords': ['рвот', 'тошнот', 'тошнит'],
                'intensity': {
                    'легкая': ['тошнот', 'подташниван'],
                    'средняя': ['рвот', 'однократн'],
                    'тяжелая': ['многократн', 'неукротим']
                }
            },
            'насморк': {
                'keywords': ['насморк', 'заложенност', 'нос', 'ринит', 'выделен', 'носа'],
                'intensity': {
                    'легкая': ['насморк', 'заложенност носа'],
                    'средняя': ['сильный насморк', 'обильные выделен'],
                    'тяжелая': ['постоянный насморк', 'непроходимост']
                }
            },
            'боль_в_груди': {
                'keywords': ['боль в груд', 'боли в груд', 'кардиалги', 'болит груд', 'боль за грудин'],
                'intensity': {
                    'легкая': ['незначительная боль', 'дискомфорт в груд'],
                    'средняя': ['боль в груд', 'боли в груд'],
                    'тяжелая': ['сильная боль', 'острая боль', 'нестерпимая боль']
                }
            }
        }
    
    def _load_medical_terms(self):
        """Медицинские термины для поиска"""
        return {
            'аускультация': ['аускультац', 'хрип', 'дыхан', 'жесткое дыхан', 'ослаблен'],
            'перкуссия': ['перкусс', 'перкуторн'],
            'пальпация': ['пальпац', 'пальпирует'],
            'отеки': ['отек', 'пастозност', 'отечност'],
            'цианоз': ['цианоз', 'синюшност', 'акроцианоз'],
            'тахикардия': ['тахикарди', 'учащен', 'сердцебиен', 'чсс'],
            'сатурация': ['сатурац', 'spo2', 'насыщен', 'кислород'],
            'артериальное_давление': ['артериальн', 'давлен', 'ад'],
            'анализ_крови': ['анализ кров', 'лейкоцит', 'гемоглобин', 'соэ', 'с-реактивн']
        }
    
    def find_symptom_intensity(self, text, symptom_name):
        """Находит интенсивность симптома в тексте"""
        if not text or not isinstance(text, str):
            return 0
            
        text_lower = text.lower()
        symptom_data = self.symptom_patterns.get(symptom_name)
        
        if not symptom_data:
            return 0
        
        # Проверяем наличие ключевых слов
        found_keywords = [kw for kw in symptom_data['keywords'] if kw in text_lower]
        if not found_keywords:
            return 0
        
        # Определяем интенсивность
        intensity = 1  # минимальная интенсивность
        
        # Безопасно проверяем интенсивности
        intensity_levels = symptom_data.get('intensity', {})
        
        # Проверяем тяжелую интенсивность (с защитой от отсутствия ключа)
        if 'тяжелая' in intensity_levels:
            for pattern in intensity_levels['тяжелая']:
                if pattern in text_lower:
                    return 3
        
        # Проверяем среднюю интенсивность (с защитой от отсутствия ключа)
        if 'средняя' in intensity_levels:
            for pattern in intensity_levels['средняя']:
                if pattern in text_lower:
                    intensity = max(intensity, 2)
        
        return intensity
    
    def find_medical_term(self, text, term_name):
        """Проверяет наличие медицинского термина"""
        if not text or not isinstance(text, str):
            return 0
            
        text_lower = text.lower()
        keywords = self.medical_terms.get(term_name, [])
        
        return 1 if any(kw in text_lower for kw in keywords) else 0
    
    def safe_get_text(self, data, *keys):
        """Безопасно извлекает текст из вложенной структуры"""
        current = data
        for key in keys:
            if isinstance(current, dict):
                current = current.get(key)
            else:
                return ''
        return str(current) if current is not None else ''
    
    def extract_text_data(self, data):
        """Извлекает текстовые данные из JSON структуры"""
        texts = []
        
        # Жалобы
        complaints_text = self.safe_get_text(data, 'complaints', 'text')
        complaints_processed = self.safe_get_text(data, 'complaints', 'processed_text')
        
        if complaints_text:
            texts.append(complaints_text)
        if complaints_processed:
            texts.append(complaints_processed)
        
        # Анамнез заболевания
        history_text = self.safe_get_text(data, 'disease_history', 'text')
        history_processed = self.safe_get_text(data, 'disease_history', 'processed_text')
        
        if history_text:
            texts.append(history_text)
        if history_processed:
            texts.append(history_processed)
        
        # Физикальное обследование
        examination_value = self.safe_get_text(data, 'physical_examination', 'value')
        examination_processed = self.safe_get_text(data, 'physical_examination', 'processed_text')
        
        if examination_value:
            texts.append(examination_value)
        if examination_processed:
            texts.append(examination_processed)
        
        return ' '.join(texts)
    
    def safe_extract_numeric(self, value_data):
        """Безопасно извлекает числовое значение"""
        if value_data is None:
            return 0
        
        value = 0
        if isinstance(value_data, dict):
            value = value_data.get('value', 0)
        elif isinstance(value_data, (int, float)):
            value = value_data
        elif isinstance(value_data, str):
            numbers = re.findall(r'\d+\.?\d*', value_data.replace(',', '.'))
            value = float(numbers[0]) if numbers else 0
        
        try:
            return float(value)
        except (ValueError, TypeError):
            return 0
    
    def extract_numeric_features(self, data):
        """Извлекает числовые параметры"""
        features = {}
        
        numeric_params = data.get('numeric_parameters', {})
        if not isinstance(numeric_params, dict):
            return features
        
        # Витальные признаки
        vital_signs = {
            'Температура тела': 'temp_body',
            'Артериальное давление систолическое': 'bp_systolic',
            'Артериальное давление диастолическое': 'bp_diastolic',
            'Частота сердечных сокращений': 'heart_rate',
            'Пульс': 'pulse',
            'Сатурация': 'saturation',
            'Частота дыхания': 'resp_rate'
        }
        
        for ru_name, en_name in vital_signs.items():
            value_data = numeric_params.get(ru_name)
            value = self.safe_extract_numeric(value_data)
            features[f'vital_{en_name}'] = value
        
        # Лабораторные показатели
        lab_tests = {
            'Лейкоциты, количество в крови методом автоматизированного подсчёта': 'wbc',
            'С-реактивный белок, массовая концентрация в сыворотке или плазме крови': 'crp',
            'Глюкоза, молярная концентрация в сыворотке или плазме крови': 'glucose',
            'Белок общий, массовая концентрация в сыворотке или плазме крови': 'total_protein'
        }
        
        for ru_name, en_name in lab_tests.items():
            value_data = numeric_params.get(ru_name)
            value = self.safe_extract_numeric(value_data)
            features[f'lab_{en_name}'] = value
        
        return features
    
    def extract_features_from_json(self, data):
        """Извлекает все признаки из JSON данных"""
        features = {}
        
        try:
            # 1. Числовые параметры
            numeric_features = self.extract_numeric_features(data)
            features.update(numeric_features)
            
            # 2. Текстовые данные
            all_text = self.extract_text_data(data)
            
            # 3. Симптомы
            for symptom_name in self.symptom_patterns.keys():
                intensity = self.find_symptom_intensity(all_text, symptom_name)
                features[f'has_{symptom_name}'] = 1 if intensity > 0 else 0
                features[f'intensity_{symptom_name}'] = intensity
            
            # 4. Медицинские термины
            for term_name in self.medical_terms.keys():
                has_term = self.find_medical_term(all_text, term_name)
                features[f'has_{term_name}'] = has_term
            
            # 5. Статистика по текстам
            complaints_text = self.safe_get_text(data, 'complaints', 'text') or self.safe_get_text(data, 'complaints', 'processed_text')
            history_text = self.safe_get_text(data, 'disease_history', 'text') or self.safe_get_text(data, 'disease_history', 'processed_text')
            examination_text = self.safe_get_text(data, 'physical_examination', 'value') or self.safe_get_text(data, 'physical_examination', 'processed_text')
            
            features['text_length_complaints'] = len(str(complaints_text))
            features['text_length_history'] = len(str(history_text))
            features['text_length_examination'] = len(str(examination_text))
            
            # 6. Сводные показатели
            symptom_names = list(self.symptom_patterns.keys())
            total_symptoms = sum(features.get(f'has_{symptom}', 0) for symptom in symptom_names)
            total_intensity = sum(features.get(f'intensity_{symptom}', 0) for symptom in symptom_names)
            
            features['total_symptoms_count'] = total_symptoms
            features['total_symptoms_intensity'] = total_intensity
            features['symptoms_diversity'] = total_symptoms / len(symptom_names) if symptom_names else 0
            
        except Exception as e:
            print(f"Ошибка при извлечении признаков: {e}")
            # Возвращаем пустые признаки при ошибке
        
        return features

def process_json_files(source_folder, target_folder):
    """Обрабатывает все JSON файлы в папке"""
    
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
        print(f"Создана папка: {target_folder}")
    
    extractor = MedicalFeatureExtractor()
    processed_files = 0
    error_files = []
    
    all_features = []
    filenames = []
    
    for filename in os.listdir(source_folder):
        if filename.endswith('.json'):
            source_path = os.path.join(source_folder, filename)
            target_path = os.path.join(target_folder, filename)
            
            try:
                with open(source_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                # Извлекаем признаки
                features = extractor.extract_features_from_json(data)
                
                # Добавляем имя файла и диагноз
                features['filename'] = filename
                
                diagnosis_data = data.get('diagnosis', {})
                if isinstance(diagnosis_data, dict):
                    features['diagnosis_code'] = diagnosis_data.get('diagnosis_text', 'Unknown')
                    features['diagnosis_name'] = diagnosis_data.get('display_name', 'Unknown')
                else:
                    features['diagnosis_code'] = 'Unknown'
                    features['diagnosis_name'] = 'Unknown'
                
                # Сохраняем обогащенный файл
                data['extracted_features'] = features
                data['feature_extraction_date'] = datetime.now().isoformat()
                
                with open(target_path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)
                
                all_features.append(features)
                filenames.append(filename)
                processed_files += 1
                
                if processed_files % 100 == 0:
                    print(f"Обработано файлов: {processed_files}")
                    
            except Exception as e:
                print(f"Ошибка при обработке файла {filename}: {e}")
                error_files.append(filename)
                # Копируем оригинальный файл при ошибке
                try:
                    shutil.copy2(source_path, target_path)
                except:
                    pass
    
    print(f"\nОбработка завершена:")
    print(f"Успешно обработано: {processed_files} файлов")
    print(f"С ошибками: {len(error_files)} файлов")
    
    return all_features, filenames, error_files

def analyze_features(features_list):
    """Анализирует извлеченные признаки"""
    if not features_list:
        print("Нет данных для анализа!")
        return None
    
    df = pd.DataFrame(features_list)
    
    print(f"\n📊 АНАЛИЗ ДАННЫХ:")
    print("=" * 60)
    print(f"Всего записей: {len(df)}")
    print(f"Всего признаков: {df.shape[1]}")
    
    # Анализ симптомов
    print(f"\n🔍 СТАТИСТИКА СИМПТОМОВ:")
    print("-" * 50)
    
    symptom_columns = [col for col in df.columns if col.startswith('has_') and any(symptom in col for symptom in ['кашель', 'температура', 'одышка', 'слабость', 'головная_боль', 'рвота', 'насморк', 'боль_в_груди'])]
    
    for symptom_col in symptom_columns:
        symptom_name = symptom_col.replace('has_', '')
        count = df[symptom_col].sum()
        percentage = (count / len(df)) * 100
        
        intensity_col = f'intensity_{symptom_name}'
        if intensity_col in df.columns:
            active_cases = df[df[symptom_col] == 1]
            if len(active_cases) > 0:
                avg_intensity = active_cases[intensity_col].mean()
                intensity_dist = active_cases[intensity_col].value_counts().sort_index()
                intensity_str = ', '.join([f"{level}:{count}" for level, count in intensity_dist.items()])
                print(f"{symptom_name:<15}: {count:4d} случаев ({percentage:5.1f}%), средняя интенсивность: {avg_intensity:.1f} [{intensity_str}]")
            else:
                print(f"{symptom_name:<15}: {count:4d} случаев ({percentage:5.1f}%)")
        else:
            print(f"{symptom_name:<15}: {count:4d} случаев ({percentage:5.1f}%)")
    
    # Топ-10 самых частых признаков
    print(f"\n🏆 ТОП-10 САМЫХ ЧАСТЫХ ПРИЗНАКОВ:")
    print("-" * 50)
    
    binary_features = [col for col in df.columns if col.startswith('has_')]
    if binary_features:
        feature_frequency = df[binary_features].sum().sort_values(ascending=False)
        
        for feature, count in feature_frequency.head(10).items():
            percentage = (count / len(df)) * 100
            print(f"{feature:<30}: {count:4d} случаев ({percentage:5.1f}%)")
    
    # Статистика по диагнозам
    if 'diagnosis_code' in df.columns:
        print(f"\n📋 РАСПРЕДЕЛЕНИЕ ДИАГНОЗОВ:")
        print("-" * 50)
        
        diagnosis_stats = df['diagnosis_code'].value_counts()
        for diagnosis, count in diagnosis_stats.head(10).items():
            percentage = (count / len(df)) * 100
            diagnosis_row = df[df['diagnosis_code'] == diagnosis].iloc[0] if not df[df['diagnosis_code'] == diagnosis].empty else None
            diagnosis_name = diagnosis_row.get('diagnosis_name', 'Unknown') if diagnosis_row is not None else 'Unknown'
            
            if diagnosis_name is None:
                diagnosis_name = 'Unknown'
            else:
                diagnosis_name = str(diagnosis_name)
            
            diagnosis_display = diagnosis_name[:40] if len(diagnosis_name) > 40 else diagnosis_name
            print(f"{diagnosis:<10}: {diagnosis_display:<40} - {count:4d} случаев ({percentage:5.1f}%)")
    
    return df

def debug_single_file(file_path):
    """Отладочная функция для одного проблемного файла"""
    print(f"\n🔍 ОТЛАДКА ФАЙЛА: {file_path}")
    print("=" * 70)
    
    extractor = MedicalFeatureExtractor()
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print("Структура JSON файла:")
        print(json.dumps(data, ensure_ascii=False, indent=2)[:1000] + "...")
        
        # Проверяем наличие ключевых разделов
        print(f"\nКлючевые разделы:")
        for section in ['complaints', 'disease_history', 'physical_examination', 'diagnosis', 'numeric_parameters']:
            exists = section in data
            print(f"  {section}: {'✅' if exists else '❌'}")
            if exists and isinstance(data[section], dict):
                print(f"    Подразделы: {list(data[section].keys())}")
        
        # Извлекаем текст
        all_text = extractor.extract_text_data(data)
        print(f"\nИзвлеченный текст (первые 500 символов):")
        print(all_text[:500] + "..." if len(all_text) > 500 else all_text)
        
        # Проверяем симптомы
        print(f"\nОбнаруженные симптомы:")
        for symptom in ['кашель', 'температура', 'одышка', 'слабость']:
            intensity = extractor.find_symptom_intensity(all_text, symptom)
            if intensity > 0:
                print(f"✅ {symptom}: интенсивность {intensity}")
            else:
                symptom_data = extractor.symptom_patterns[symptom]
                found_kw = [kw for kw in symptom_data['keywords'] if kw in all_text.lower()]
                print(f"❌ {symptom}: ключевые слова {found_kw if found_kw else 'не найдены'}")
                
    except Exception as e:
        print(f"Ошибка при анализе файла: {e}")
        import traceback
        traceback.print_exc()

def main():
    source_folder = r"top9_merged_diagnoses"
    enriched_folder = "enriched_medical_results_corrected"
    
    # Сначала проверяем проблемный файл
    problem_file = os.path.join(source_folder, "processed_Эпикриз_1505_v1.json")
    if os.path.exists(problem_file):
        debug_single_file(problem_file)
    
    # Обрабатываем файлы
    print(f"\n🎯 Обработка JSON файлов...")
    features_list, filenames, error_files = process_json_files(source_folder, enriched_folder)
    
    # Анализируем результаты
    df = analyze_features(features_list)
    
    if df is not None:
        # Сохраняем CSV
        output_path = "medical_features_analysis.csv"
        df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"\n💾 Данные сохранены в: {output_path}")
        
        # Показываем пример извлеченных признаков
        print(f"\n🔬 ПРИМЕР ИЗВЛЕЧЕННЫХ ПРИЗНАКОВ:")
        print("-" * 50)
        if len(df) > 0:
            sample_features = df.iloc[0]
            
            # Активные симптомы
            active_symptoms = []
            symptom_cols = [col for col in df.columns if col.startswith('has_') and any(s in col for s in ['кашель', 'температура', 'одышка', 'слабость', 'головная_боль', 'рвота', 'насморк', 'боль_в_груди'])]
            
            for col in symptom_cols:
                if sample_features[col] == 1:
                    symptom_name = col.replace('has_', '')
                    active_symptoms.append(symptom_name)
            
            print(f"Активные симптомы: {', '.join(active_symptoms) if active_symptoms else 'нет'}")
            
            # Интенсивности
            if active_symptoms:
                print("Интенсивности:")
                for symptom in active_symptoms:
                    intensity = sample_features.get(f'intensity_{symptom}', 0)
                    levels = {1: 'легкая', 2: 'средняя', 3: 'тяжелая'}
                    level = levels.get(intensity, 'отсутствует')
                    print(f"  {symptom}: {intensity} ({level})")

if __name__ == "__main__":
    main()

# Считывание в датафрейм

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

# Загрузка данных
df = pd.read_csv('medical_features_analysis.csv')
columns_to_skip = ['text_length_complaints', 'text_length_history', 'text_length_examination']
df_filtered = df.drop(columns=[col for col in columns_to_skip if col in df.columns], errors='ignore')

print(f"📊 Всего записей: {len(df_filtered)}")
print(f"📋 Столбцы: {list(df_filtered.columns)}")

# Добавляем CSS для горизонтальной прокрутки
display(HTML("""
<style>
.dataframe-container {
    overflow-x: auto;
    max-width: 100%;
    border: 1px solid #ccc;
    margin: 10px 0;
}
.dataframe {
    min-width: 100%;
}
</style>
"""))

# Создаем виджеты для навигации
page_size = widgets.IntSlider(value=10, min=5, max=50, step=5, description='Строк на страницу:')
page_number = widgets.IntSlider(value=0, min=0, max=max(0, (len(df_filtered)-1)//10), description='Страница:')

def display_page(page=0, size=10):
    start_idx = page * size
    end_idx = min(start_idx + size, len(df_filtered))
    
    if start_idx >= len(df_filtered):
        print("Нет данных для отображения")
        return
    
    # Отображаем с CSS оберткой для прокрутки
    display(HTML(f'<div class="dataframe-container">{df_filtered.iloc[start_idx:end_idx].to_html()}</div>'))

# Создаем интерактивный виджет
interactive_display = widgets.interactive(display_page, page=page_number, size=page_size)
display(interactive_display)

# Анализ корреляции

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Загрузка данных
df = pd.read_csv('medical_features_analysis.csv')
columns_to_skip = ['text_length_complaints', 'text_length_history', 'text_length_examination']
df_filtered = df.drop(columns=[col for col in columns_to_skip if col in df.columns], errors='ignore')

print(f"📊 Всего записей: {len(df_filtered)}")
print(f"📋 Все столбцы: {list(df_filtered.columns)}")

# Исключаем filename и diagnosis_name из анализа корреляции
columns_to_exclude = ['filename', 'diagnosis_name']
numeric_columns = [col for col in df_filtered.columns 
                  if col not in columns_to_exclude 
                  and pd.api.types.is_numeric_dtype(df_filtered[col])]

print(f"📈 Столбцы для анализа корреляции ({len(numeric_columns)}):")
for col in numeric_columns:
    print(f"  - {col}")

# Создаем DataFrame только с числовыми столбцами для корреляции
df_corr = df_filtered[numeric_columns]

# 1. Матрица корреляции
correlation_matrix = df_corr.corr()
print("\n🔍 Матрица корреляции:")
# Отображаем матрицу корреляции без стилизации
with pd.option_context('display.precision', 3):
    display(correlation_matrix)

# 2. Визуализация тепловой карты корреляции
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, 
            mask=mask,
            annot=True, 
            cmap='coolwarm', 
            center=0,
            fmt='.3f',
            square=True,
            cbar_kws={'shrink': 0.8},
            annot_kws={'size': 8})
plt.title('Тепловая карта корреляции числовых признаков', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 3. Сильные корреляции (по абсолютному значению)
print("\n🔥 САМЫЕ СИЛЬНЫЕ КОРРЕЛЯЦИИ:")
print("=" * 60)

strong_correlations = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_value = correlation_matrix.iloc[i, j]
        if abs(corr_value) > 0.5:
            strong_correlations.append({
                'Признак 1': correlation_matrix.columns[i],
                'Признак 2': correlation_matrix.columns[j],
                'Корреляция': corr_value
            })

if strong_correlations:
    strong_corr_df = pd.DataFrame(strong_correlations)
    strong_corr_df = strong_corr_df.sort_values('Корреляция', key=abs, ascending=False)
    
    print(f"Найдено {len(strong_correlations)} сильных корреляций (|r| > 0.5):")
    print("-" * 60)
    for idx, row in strong_corr_df.iterrows():
        corr_type = "положительная" if row['Корреляция'] > 0 else "отрицательная"
        print(f"{idx+1:2d}. {row['Признак 1']:20} <-> {row['Признак 2']:20} | r = {row['Корреляция']:7.3f} ({corr_type})")
else:
    print("Нет сильных корреляций (|r| > 0.5)")

# 4. Статистика по корреляциям
print("\n📊 СТАТИСТИКА КОРРЕЛЯЦИЙ:")
print("=" * 40)

corr_values = correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)]

if len(corr_values) > 0:
    stats = [
        ('Минимальная корреляция', f"{corr_values.min():.3f}"),
        ('Максимальная корреляция', f"{corr_values.max():.3f}"),
        ('Средняя корреляция (по модулю)', f"{np.abs(corr_values).mean():.3f}"),
        ('Медианная корреляция', f"{np.median(corr_values):.3f}"),
        ('Стандартное отклонение', f"{corr_values.std():.3f}"),
        ('Количество |r| > 0.7', f"{np.sum(np.abs(corr_values) > 0.7)}"),
        ('Количество |r| > 0.5', f"{np.sum(np.abs(corr_values) > 0.5)}"),
        ('Количество |r| > 0.3', f"{np.sum(np.abs(corr_values) > 0.3)}")
    ]
    
    for name, value in stats:
        print(f"{name:35} : {value}")
else:
    print("Недостаточно данных для статистики")

# 5. Топ-10 самых сильных корреляций (включая слабые)
print("\n🏆 ТОП-10 КОРРЕЛЯЦИЙ (по абсолютному значению):")
print("=" * 70)

all_correlations = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_value = correlation_matrix.iloc[i, j]
        all_correlations.append({
            'Признак 1': correlation_matrix.columns[i],
            'Признак 2': correlation_matrix.columns[j],
            'Корреляция': corr_value
        })

if all_correlations:
    all_corr_df = pd.DataFrame(all_correlations)
    top_10_corr = all_corr_df.reindex(all_corr_df['Корреляция'].abs().sort_values(ascending=False).index).head(10)
    
    for idx, row in top_10_corr.iterrows():
        corr_type = "🟥 ПОЛОЖИТЕЛЬНАЯ" if row['Корреляция'] > 0 else "🟦 ОТРИЦАТЕЛЬНАЯ"
        strength = "ОЧЕНЬ СИЛЬНАЯ" if abs(row['Корреляция']) > 0.7 else \
                  "СИЛЬНАЯ" if abs(row['Корреляция']) > 0.5 else \
                  "УМЕРЕННАЯ" if abs(row['Корреляция']) > 0.3 else "СЛАБАЯ"
        
        print(f"{idx+1:2d}. {row['Признак 1'][:25]:25} <-> {row['Признак 2'][:25]:25}")
        print(f"     r = {row['Корреляция']:7.3f} | {corr_type} | {strength}")
        print()

# 6. Попарные графики для топ-4 сильно коррелирующих признаков
if strong_correlations:
    print("\n📈 ГРАФИКИ СИЛЬНО КОРРЕЛИРУЮЩИХ ПРИЗНАКОВ:")
    top_pairs = strong_corr_df.head(4)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.ravel()
    
    for idx, (_, pair) in enumerate(top_pairs.iterrows()):
        col1 = pair['Признак 1']
        col2 = pair['Признак 2']
        corr_val = pair['Корреляция']
        
        # Удаляем NaN значения для построения графиков
        valid_data = df_corr[[col1, col2]].dropna()
        
        if len(valid_data) > 0:
            axes[idx].scatter(valid_data[col1], valid_data[col2], alpha=0.6, s=50)
            axes[idx].set_xlabel(col1, fontsize=10)
            axes[idx].set_ylabel(col2, fontsize=10)
            axes[idx].set_title(f'{col1} vs {col2}\n(r = {corr_val:.3f})', fontsize=12)
            
            # Добавляем линию тренда
            if len(valid_data) > 1:
                z = np.polyfit(valid_data[col1], valid_data[col2], 1)
                p = np.poly1d(z)
                x_range = np.linspace(valid_data[col1].min(), valid_data[col1].max(), 100)
                axes[idx].plot(x_range, p(x_range), "r--", alpha=0.8, linewidth=2)
        
        # Скрываем пустые subplots
        if idx >= len(top_pairs):
            axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

# 7. Информация о пропущенных значениях
print("\n🔍 ИНФОРМАЦИЯ О ДАННЫХ:")
print("=" * 40)
print(f"Всего числовых признаков: {len(numeric_columns)}")
print(f"Всего записей: {len(df_corr)}")
print(f"Записей без пропусков: {len(df_corr.dropna())}")
print(f"Процент пропусков: {(1 - len(df_corr.dropna()) / len(df_corr)) * 100:.1f}%")

# Пропуски по столбцам
print("\nПропуски по столбцам:")
for col in numeric_columns:
    missing = df_corr[col].isna().sum()
    if missing > 0:
        print(f"  - {col}: {missing} пропусков ({missing/len(df_corr)*100:.1f}%)")